In [6]:
# from google.colab import drive
# drive.mount('/content/drive')

In [7]:
!pip uninstall -y torch torchvision torchaudio

!pip install torch==2.4.0+cu118 torchvision==0.19.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118

!pip install transformers==4.38.2

Found existing installation: torch 2.4.0+cu118
Uninstalling torch-2.4.0+cu118:
  Successfully uninstalled torch-2.4.0+cu118
Found existing installation: torchvision 0.19.0+cu118
Uninstalling torchvision-0.19.0+cu118:
  Successfully uninstalled torchvision-0.19.0+cu118
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
  Using cached https://download-r2.pytorch.org/whl/cu118/torch-2.4.0%2Bcu118-cp312-cp312-linux_x86_64.whl (857.7 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.19.0%2Bcu118-cp312-cp312-linux_x86_64.whl (6.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] [torchvision]


In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
DATASET = "gsm"
INPUT_PATH = Path(
    "/kaggle/input/datasets/kaungmyatkyaw/compression-dataset/gsm_llmlingua2_results.json"
)
OUTPUT_PATH = Path(
    "/kaggle/working/gsm_responses.json"
)
MODEL_NAME = "GSAI-ML/LLaDA-8B-Instruct"

MAX_ITEMS = 250

GEN_KWARGS = dict(
    steps=128,
    gen_length=128,
    block_length=32,
    temperature=0.0,
    cfg_scale=0.0
)
MASK_ID = 126336
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SUMMARIZATION_DATASETS = {
    "duc2004",
    "bnc",
    "google",
    "broadcast"
}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.padding_side != "left":
    tokenizer.padding_side = "left"

model = (
    AutoModel.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16
    )
    .to(DEVICE)
    .eval()
)

print("Model loaded.")

In [ ]:
def add_gumbel_noise(logits, temperature):
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel_noise = (- torch.log(noise)) ** temperature
    return logits.exp() / gumbel_noise


def get_num_transfer_tokens(mask_index, steps):
    mask_num = mask_index.sum(dim=1, keepdim=True)
    base = mask_num // steps
    remainder = mask_num % steps
    num_transfer_tokens = (
        torch.zeros(mask_num.size(0), steps, device=mask_index.device, dtype=torch.int64) + base
    )
    for i in range(mask_num.size(0)):
        num_transfer_tokens[i, :remainder[i]] += 1
    return num_transfer_tokens


@torch.no_grad()
def generate(model, prompt, attention_mask=None, steps=128, gen_length=128, block_length=32,
             temperature=0., cfg_scale=0., remasking='low_confidence', mask_id=MASK_ID):
    x = torch.full((prompt.shape[0], prompt.shape[1] + gen_length), mask_id, dtype=torch.long).to(model.device)
    x[:, :prompt.shape[1]] = prompt.clone()

    if attention_mask is not None:
        attention_mask = torch.cat([
            attention_mask,
            torch.ones((prompt.shape[0], gen_length), dtype=attention_mask.dtype, device=model.device)
        ], dim=-1)

    prompt_index = (x != mask_id)

    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    assert steps % num_blocks == 0
    steps = steps // num_blocks

    for num_block in range(num_blocks):
        block_mask_index = (
            x[:, prompt.shape[1] + num_block * block_length:
                  prompt.shape[1] + (num_block + 1) * block_length] == mask_id
        )
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps)

        for i in range(steps):
            mask_index = (x == mask_id)

            if cfg_scale > 0.:
                un_x = x.clone()
                un_x[prompt_index] = mask_id
                x_ = torch.cat([x, un_x], dim=0)
                if attention_mask is not None:
                    attention_mask_ = torch.cat([attention_mask, attention_mask], dim=0)
                logits = model(x_, attention_mask=attention_mask_).logits
                logits, un_logits = torch.chunk(logits, 2, dim=0)
                logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
            else:
                logits = model(x, attention_mask=attention_mask).logits

            logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
            x0 = torch.argmax(logits_with_noise, dim=-1)

            if remasking == 'low_confidence':
                p = F.softmax(logits, dim=-1)
                x0_p = torch.squeeze(
                    torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1)
            elif remasking == 'random':
                x0_p = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
            else:
                raise NotImplementedError(remasking)

            x0_p[:, prompt.shape[1] + (num_block + 1) * block_length:] = -np.inf
            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, x0_p, -np.inf)

            transfer_index = torch.zeros_like(x0, dtype=torch.bool, device=x0.device)
            for j in range(confidence.shape[0]):
                _, select_index = torch.topk(confidence[j], k=num_transfer_tokens[j, i])
                transfer_index[j, select_index] = True
            x[transfer_index] = x0[transfer_index]

    return x

In [ ]:
def generate_one(prompt_text: str, steps=128, gen_length=128, block_length=32,
                 temperature=0.0, cfg_scale=0.0) -> str:
    message = {"role": "user", "content": prompt_text}
    chat_prompt = tokenizer.apply_chat_template(
        [message], add_generation_prompt=True, tokenize=False
    )
    encoded = tokenizer(chat_prompt, add_special_tokens=False, return_tensors="pt")
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    out = generate(
        model, input_ids, attention_mask,
        steps=steps, gen_length=gen_length, block_length=block_length,
        temperature=temperature, cfg_scale=cfg_scale, remasking='low_confidence'
    )
    response = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

In [ ]:
def build_maths_prompt(text):
    return (
        "Your answer should only contain a number. "
        f"Answer the question: {text}"
    )


def build_reconstruction_prompt(text):
    return (
        'Your answer should start with "Answer: ". '
        f"Recover the original text from this compressed version: {text}"
    )


def build_summary_prompt(text):
    return (
        'Your answer should start with "Summary: ". '
        f"Summarise the following text:\n\n{text}"
    )


def extract_after(text, prefix):
    if prefix in text:
        idx = text.index(prefix)
        return text[idx + len(prefix):].strip()
    return ""


def extract_gsm_number(text):
    match = re.search(r"####\s*([\d,]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    nums = re.findall(r"[\d,]+", text)
    return nums[-1].replace(",", "") if nums else ""


with open(INPUT_PATH, encoding="utf-8") as f:
    data = json.load(f)

responses = []

if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, encoding="utf-8") as f:
        responses = json.load(f)

done_indices = {item["index"] for item in responses}

print(f"Loaded dataset: {len(data)} samples")

In [ ]:
processed = 0

for item in tqdm(data):

    idx = item["index"]

    if idx in done_indices:
        print(f"Skipping {idx} (already done)")
        continue

    if processed >= MAX_ITEMS:
        print(f"Reached {MAX_ITEMS} new items.")
        break

    print(f"\nProcessing index {idx}")

    metadata = item.get("metadata", {})
    result_metadata = item.get("result", {})

    original_prompt = item.get("prompt", "")

    compressed_prompt = (
        item.get("compressed_prompt")
        or result_metadata.get("compressed_prompt", "")
    )

    if DATASET == "gsm":

        answer_text = (
            metadata.get("answer")
            or item.get("answer", "")
        )

        ground_truth = extract_gsm_number(answer_text)

        original_response = generate_one(
            build_maths_prompt(original_prompt),
            **GEN_KWARGS
        )

        compressed_response = generate_one(
            build_maths_prompt(compressed_prompt),
            **GEN_KWARGS
        )

        reconstruction_response = generate_one(
            build_reconstruction_prompt(compressed_prompt),
            **GEN_KWARGS
        )

        new_item = {
            "index": idx,
            "prompt": original_prompt,
            "metadata": metadata,
            "ground_truth": ground_truth,
            "result": result_metadata,
            "prompt_response": original_response,
            "compressed_prompt_response": compressed_response,
            "reconstruction_response": reconstruction_response
        }

    elif DATASET in SUMMARIZATION_DATASETS:

        original_response = generate_one(
            build_summary_prompt(original_prompt),
            **GEN_KWARGS
        )

        compressed_response = generate_one(
            build_summary_prompt(compressed_prompt),
            **GEN_KWARGS
        )

        new_item = {
            "index": idx,
            "prompt": original_prompt,
            "metadata": metadata,
            "result": result_metadata,
            "prompt_response": original_response,
            "compressed_prompt_response": compressed_response
        }

    else:
        raise ValueError(f"Unsupported dataset: {DATASET}")

    responses.append(new_item)

    processed += 1

    done_indices.add(idx)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(
            responses,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\n=== ORIGINAL PROMPT ===")
    print(original_prompt)

    print("\n=== COMPRESSED PROMPT ===")
    print(compressed_prompt)

    print("\n=== ORIGINAL RESPONSE ===")
    print(original_response)

    print("\n=== COMPRESSED RESPONSE ===")
    print(compressed_response)

    if DATASET == "gsm":

        print("\n=== RECONSTRUCTION RESPONSE ===")
        print(reconstruction_response)

        print("\n=== GROUND TRUTH ===")
        print(ground_truth)

print(f"\nSaved results to: {OUTPUT_PATH}")
print(f"Processed this run: {processed}")
print(f"Total saved samples: {len(responses)}")